# 01.02 · Messages and standard content blocks

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lesson 01.01. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** conversations with `SystemMessage`, `HumanMessage`, `AIMessage` and `ToolMessage`, and with the equivalent dict and tuple formats.
2. **Read** every part of an `AIMessage`: `text`, `content`, `content_blocks`, `tool_calls`, `usage_metadata`, `response_metadata`, `id`.
3. **Explain** `content` (provider-shaped) vs `content_blocks` (standard, provider-agnostic) and write one loop that reads reasoning, text and tool calls from any provider.
4. **Send** an image to a vision model with a standard image content block, knowing the provider's limits.
5. **Serialize** a conversation to JSON and back, and **render** messages with rich display.

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [Messages](https://docs.langchain.com/oss/python/langchain/messages) | **Main page.** Types, formats, content, standard content blocks, multimodal, block reference |
| [What's new in LangChain v1 → standard content blocks](https://docs.langchain.com/oss/python/releases/langchain-v1) | Why content blocks exist; "backward compatible, loaded lazily" |
| [Groq reasoning](https://console.groq.com/docs/reasoning) | `reasoning_format`, `include_reasoning`, `reasoning_effort` per model |
| [Groq images and vision](https://console.groq.com/docs/vision) | The vision model, size and image-count limits, token cost per image |
| [`AIMessage` reference](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage) · [`UsageMetadata`](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata) | Every attribute |
| [Block translators (source code)](https://github.com/langchain-ai/langchain/tree/master/libs/core/langchain_core/messages/block_translators) | How `content_blocks` is computed per provider (research checkpoint) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "pandas>=3.0.6,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Four message types

A chat model doesn't receive "a prompt"; it receives a **list of messages**, each with a **role**. LangChain has one class per role ([Messages → Message types](https://docs.langchain.com/oss/python/langchain/messages#message-types)):

| Class | Role sent to the provider | Who writes it | Purpose |
|---|---|---|---|
| `SystemMessage` | `system` | You (the developer) | Rules, persona, output format. Highest-priority instructions |
| `HumanMessage` | `user` | The end user (or you, in tests) | The request; may contain images and files |
| `AIMessage` | `assistant` | The model (or you, to replay history) | The answer, reasoning, tool calls, usage |
| `ToolMessage` | `tool` | Your code, after running a tool | A tool's result, linked by `tool_call_id` |

Import them from `langchain.messages` (the v1 public path; they are defined in `langchain_core.messages`).

In [ ]:
# 🧪 Offline: build a small conversation and pretty-print it.
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

conversation = [
    SystemMessage("You are a concise tutor. Answer in one sentence."),
    HumanMessage("What is a token?"),
    AIMessage("A token is a chunk of text, often part of a word, that a model reads and writes."),
    HumanMessage("And roughly how many tokens is one English word?"),
]
for message in conversation:
    message.pretty_print()

**You should now see** four blocks with headers like `================ System Message ================`. `pretty_print()` is a quick terminal-style view; Part 3 shows a richer HTML one.

---
## Part 2 · Same conversation, four formats

LangChain accepts several input shapes and **normalizes** them to message objects before calling the provider:

| Format | Example | Where you meet it |
|---|---|---|
| Message objects | `HumanMessage("hi")` | Library code; type-safe; IDE autocompletion |
| Dicts (OpenAI chat format) | `{"role": "user", "content": "hi"}` | JSON from a web front end, datasets, LangGraph state examples |
| Tuples | `("human", "hi")` | Short examples and prompt templates (Lesson 01.03) |
| A plain string | `"hi"` | Shortcut for **one** `HumanMessage` |

`convert_to_messages` is the function LangChain uses internally. Let's call it ourselves.

In [ ]:
# 🧪 Offline: three formats in, identical message objects out.
from langchain_core.messages import convert_to_messages

as_dicts = [{"role": "system", "content": "Be concise."}, {"role": "user", "content": "Hi!"}]
as_tuples = [("system", "Be concise."), ("human", "Hi!")]
as_objects = [SystemMessage("Be concise."), HumanMessage("Hi!")]

for label, value in (("dicts", as_dicts), ("tuples", as_tuples), ("objects", as_objects)):
    converted = convert_to_messages(value)
    print(f"{label:<8} → {[type(m).__name__ for m in converted]}  same content: "
          f"{[m.content for m in converted] == ['Be concise.', 'Hi!']}")

### What just happened?

All three became `[SystemMessage, HumanMessage]` with the same content. **Course rule:** objects in Python code, dicts at the edges (HTTP payloads, JSON files). Mixing both in one list also works, but it makes code harder to read.

---
## Part 3 · Anatomy of an `AIMessage`

Below is an `AIMessage` built by hand with the **same fields `ChatGroq` fills in** for a `gpt-oss` answer (we copied the shape from a real response). Building it offline lets every learner inspect it, key or not.

In [ ]:
# 🧪 Offline: an AIMessage shaped exactly like a ChatGroq response from a gpt-oss model.
groq_style = AIMessage(
    content="About 0.75 words per token, so roughly 1.3 tokens per English word.",
    additional_kwargs={"reasoning_content": "User asks tokens per word. Rule of thumb: ~4 chars per token."},
    response_metadata={
        "model_provider": "groq", "model_name": "openai/gpt-oss-20b", "finish_reason": "stop",
        "token_usage": {"prompt_tokens": 92, "completion_tokens": 41, "total_tokens": 133},
    },
    usage_metadata={"input_tokens": 92, "output_tokens": 41, "total_tokens": 133,
                    "output_token_details": {"reasoning": 19}},
    id="lc_run--demo-0001",
)
for attribute in ("text", "content", "additional_kwargs", "usage_metadata", "id"):
    print(f"{attribute:<18}: {getattr(groq_style, attribute)!r}")
print(f"{'response_metadata':<18}: {groq_style.response_metadata!r}")

In [ ]:
# 🧪 Rich display: the course helper renders the important parts as an HTML card (all text escaped).
from mastery.display import show_message

show_message(groq_style)

### What just happened?

| Attribute | What it holds | Standard across providers? |
|---|---|---|
| `.text` | The answer text only (a property; joins text blocks) | ✅ |
| `.content` | The **raw** payload: a string here; a list of dicts for some providers | ❌ provider-shaped |
| `.additional_kwargs` | Provider extras. Groq puts reasoning in `reasoning_content` | ❌ |
| `.response_metadata` | Model name, finish reason, raw token usage, `model_provider` | Partly (keys vary) |
| `.usage_metadata` | Token counts in LangChain's **standard** shape (Lesson 01.06) | ✅ |
| `.tool_calls` | Parsed tool calls (empty here) | ✅ |
| `.id` | Message ID (LangChain's `lc_run--...` or the provider's) | ✅ |

`show_message` is ~80 lines in [`src/mastery/display.py`](../../src/mastery/display.py). It uses `IPython.display.HTML`, and escapes every string so model output can never inject HTML or scripts into your notebook. You'll use it in the rest of the course.

---
## Part 4 · `content` vs `content_blocks`: the provider-agnostic view

Providers return reasoning in different places: Groq in `additional_kwargs["reasoning_content"]`, Anthropic as a `thinking` block inside `content`, OpenAI's Responses API as `reasoning` blocks with `summary` lists. Reading each one by hand means provider-specific `if`s everywhere.

**`content_blocks`** (new in LangChain v1) is a property that **lazily translates** `content` + `additional_kwargs` into **standard typed blocks**: `text`, `reasoning`, `tool_call`, `image`, `server_tool_call`, ... ([Messages → Standard content blocks](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks)). The translation uses `response_metadata["model_provider"]` to pick the right translator.

In [ ]:
# 🧪 Offline: three providers, three raw shapes (examples copied from the Messages page + Part 3).
anthropic_style = AIMessage(
    content=[{"type": "thinking", "thinking": "Rule of thumb: ~4 characters per token.", "signature": "sig123"},
             {"type": "text", "text": "Roughly 1.3 tokens per English word."}],
    response_metadata={"model_provider": "anthropic"},
)
openai_style = AIMessage(
    content=[{"type": "reasoning", "id": "rs_1", "summary": [{"type": "summary_text", "text": "~4 chars per token."}]},
             {"type": "text", "text": "Roughly 1.3 tokens per English word.", "id": "msg_1"}],
    response_metadata={"model_provider": "openai"},
)
for name, message in (("groq", groq_style), ("anthropic", anthropic_style), ("openai", openai_style)):
    print(f"{name:<10} content type: {type(message.content).__name__}")

In [ ]:
# 🧪 ONE loop for all three providers, thanks to content_blocks.
def describe_blocks(message: AIMessage) -> list[str]:
    """Return one short line per standard content block."""
    lines = []
    for block in message.content_blocks:
        if block["type"] == "reasoning":
            lines.append(f"🧠 reasoning: {block.get('reasoning', '')[:45]}")
        elif block["type"] == "text":
            lines.append(f"💬 text     : {block['text'][:45]}")
        elif block["type"] == "tool_call":
            lines.append(f"🔧 tool_call: {block['name']}({block['args']})")
        else:
            lines.append(f"• {block['type']}")
    return lines

for name, message in (("groq", groq_style), ("anthropic", anthropic_style), ("openai", openai_style)):
    print(name, *describe_blocks(message), sep="\n  ")

### What just happened?

- Raw `content` was a `str` for Groq and a `list` for the other two, with different keys (`thinking` vs `reasoning`/`summary`).
- `content_blocks` gave the **same shape** for all three: a `reasoning` block then a `text` block. `describe_blocks` has no provider-specific code.
- Nothing was converted in place: `content` is unchanged (backward compatible). `content_blocks` is computed when you read it.

> **Rule of thumb:** use `.text` when you only need the answer, `content_blocks` when you need reasoning, citations, tool calls or multimodal parts, and `content` only when you need a provider-specific detail that the standard blocks don't carry (it may be under `extras`).

---
## Part 5 · Tool calls and `ToolMessage` (preview of Module 02)

When a model decides to use a tool, its `AIMessage` contains **`tool_calls`** instead of (or in addition to) text. Your code runs the tool and answers with a **`ToolMessage`** whose `tool_call_id` **must match** the call's `id`; that's how the model pairs results with requests when it made several calls. Agents do this loop for you in Module 02; here you build it by hand to see the structure.

In [ ]:
# 🧪 Offline: a tool-calling round trip, built by hand.
ai_calls_tool = AIMessage(
    content="",
    tool_calls=[{"name": "get_weather", "args": {"city": "Bogotá"}, "id": "call_123"}],
)
tool_result = ToolMessage(
    content="17°C, light rain",
    tool_call_id="call_123",             # Must equal the id above.
    name="get_weather",
    artifact={"source": "demo-weather-api", "raw": {"temp_c": 17.2}},   # Kept in code, NOT sent to the model.
)
history = [HumanMessage("Weather in Bogotá?"), ai_calls_tool, tool_result]
print("tool_calls     :", ai_calls_tool.tool_calls)
print("content_blocks :", ai_calls_tool.content_blocks)
print("ToolMessage    :", tool_result.content, "| artifact:", tool_result.artifact)

### What just happened?

- `tool_calls` gained `"type": "tool_call"`, and `content_blocks` lists the same call as a standard block.
- `artifact` holds data **for your code** (raw API response, document IDs) that is **not** sent to the model, so it doesn't cost tokens or leak into the prompt ([Messages → Tool message](https://docs.langchain.com/oss/python/langchain/messages#tool-message)).
- The next step would be `model.invoke(history)`: the model reads the tool result and writes the final answer. That's an agent loop; `create_agent` automates it in Lesson 02.02.

---
## Part 6 · Multi-turn conversations: the model remembers nothing

Chat model APIs are **stateless**: each call must include the whole conversation you want the model to consider. "Memory" is just *you* appending messages to a list and sending it again. Two consequences: every turn costs more input tokens than the last, and a conversation can eventually exceed the context window (Module 03 handles both).

In [ ]:
# 💸 Calls Groq twice (fast model), ~400 tokens. The 2nd question only makes sense with history.
from mastery.models import get_model

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm = get_model("fast", temperature=0, max_tokens=300, reasoning_effort="low")
    chat = [SystemMessage("You are a concise tutor. One sentence per answer."),
            HumanMessage("Name one advantage of open-weight models.")]
    first = llm.invoke(chat)
    chat += [first, HumanMessage("Now give one DISadvantage of that same thing.")]
    second = llm.invoke(chat)
    for m in chat[1:] + [second]:
        print(f"{m.type:>5}: {m.text}")
    print("\ninput tokens, turn 1 → turn 2:", first.usage_metadata["input_tokens"], "→", second.usage_metadata["input_tokens"])

**You should now see** the follow-up answer refer to the first one ("that same thing"), and **input tokens grow** from turn 1 to turn 2, because turn 2 resends everything.

> Notice that we appended the model's own `AIMessage` (`first`) to the history. That keeps reasoning and metadata attached. Rebuilding it as `AIMessage(first.text)` also works but drops them.

---
## Part 7 · Serializing a conversation

To store a chat (database, file, API response) you need plain JSON. Two helpers from `langchain_core.messages`:

- `messages_to_dict` / `messages_from_dict`: a **lossless** LangChain round trip (keeps types, IDs, metadata).
- `convert_to_openai_messages`: the widely understood `{"role", "content"}` format, for other systems. **Lossy:** LangChain-specific metadata is dropped.

In [ ]:
# 🧪 Offline: lossless round trip through JSON, then the portable OpenAI format.
import json

from langchain_core.messages import convert_to_openai_messages, messages_from_dict, messages_to_dict

as_json = json.dumps(messages_to_dict(history), ensure_ascii=False)
restored = messages_from_dict(json.loads(as_json))
print("JSON size:", len(as_json), "characters")
print("Round trip equal:", restored == history)
print(json.dumps(convert_to_openai_messages(history), indent=1, ensure_ascii=False)[:600])

### What just happened?

- `restored == history` is `True`: types, tool calls and the artifact survived.
- The OpenAI format shows `"role": "assistant"` with `tool_calls` and `"role": "tool"` with `tool_call_id`, the shape most chat APIs and datasets use. The artifact is gone (it's LangChain-only).
- `ensure_ascii=False` keeps "Bogotá" readable instead of `\u00e1` escapes.

---
## Part 8 · Multimodal input: sending an image

A `HumanMessage` can carry a **list** of content blocks: text plus images, audio, video or files. The **standard** image block looks like this ([Messages → Multimodal](https://docs.langchain.com/oss/python/langchain/messages#multimodal)):

```python
{"type": "image", "base64": "<data>", "mime_type": "image/png"}   # or {"type": "image", "url": "https://..."}
```

The integration converts it to the provider's format (for Groq: an OpenAI-style `image_url` with a `data:` URL). Groq's vision model, verified 2026-10-11 on [Groq vision](https://console.groq.com/docs/vision):

| Fact | Value |
|---|---|
| Model | `qwen/qwen3.8-27b`, a **preview** model (`MASTERY_MODEL_VISION`) |
| Images per request | at most **3** |
| Cost | each image counts as **2,048 input tokens** |
| Size | 20 MB max for a request with an image URL |
| Not supported by `ChatGroq` | audio, video and PDF input (see the [feature table](https://docs.langchain.com/oss/python/integrations/chat/groq)); Gemini handles those (Modules 06, 11, 13) |

First, look at the image with rich display (`IPython.display.Image`).

In [ ]:
# 🧪 Offline: load a synthetic chart from data/samples and show it inline.
import base64

from IPython.display import Image, display

IMAGE_PATH = REPO_ROOT / "data" / "samples" / "images" / "quarterly-signups.png"
image_bytes = IMAGE_PATH.read_bytes()
display(Image(data=image_bytes, width=380))
print(f"{IMAGE_PATH.name}: {len(image_bytes):,} bytes")

In [ ]:
# 🧪 Offline: build the multimodal message with STANDARD blocks (no provider-specific format).
image_message = HumanMessage(content=[
    {"type": "text", "text": "Which quarter had the most sign-ups, and which one dipped? Answer in one sentence."},
    {"type": "image", "base64": base64.b64encode(image_bytes).decode("ascii"), "mime_type": "image/png"},
])
print([block["type"] for block in image_message.content_blocks])
print("base64 length:", len(image_message.content_blocks[1]["base64"]), "characters (~33% bigger than the file)")

In [ ]:
# 💸 Calls Groq's PREVIEW vision model, ~2.2k input tokens (2,048 for the image).
import warnings

from mastery.models import PreviewModelWarning

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", PreviewModelWarning)   # We know; see the Production note below.
        vision = get_model("vision", temperature=0, max_tokens=400, reasoning_effort="none")
    answer = vision.invoke([image_message])
    show_message(answer)

### What just happened?

- `content_blocks` showed `['text', 'image']`. The same message would work with an OpenAI or Gemini vision model, because you used standard blocks.
- With a key: the answer should name **Q4** as the highest (240) and **Q3** as the dip (150), and `input_tokens` should be a bit over 2,048.
- `reasoning_effort="none"`: Groq lets you switch Qwen's reasoning off (values `none`, `default`, `low`, `medium`, `high` per [Groq reasoning](https://console.groq.com/docs/reasoning)). For a simple visual question, thinking would only add tokens.

> ⚠️ **Production note: base64 images in requests.**
> Base64 makes payloads ~33% larger and every image costs 2,048 tokens on Groq. In production: resize images before sending, prefer provider file storage or URLs for repeated images, validate MIME type and size on upload, and never send images users didn't consent to process. The vision model here is a **preview** model: evaluation only.

---
## Part 9 · Storing standard blocks in `content`: `output_version="v1"`

By default, `content` stays provider-shaped and `content_blocks` is computed on read. If code **outside** LangChain (a front end, a database consumer) needs the standard blocks, ask the model to store them directly: `init_chat_model(..., output_version="v1")` or the environment variable `LC_OUTPUT_VERSION=v1` ([Messages → Serializing standard content](https://docs.langchain.com/oss/python/langchain/messages#standard-content-blocks)).

In [ ]:
# 💸 Calls Groq (fast model), ~150 tokens. Compare the raw `content` type.
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    v1_model = get_model("fast", temperature=0, max_tokens=200, reasoning_effort="low", output_version="v1")
    v1_answer = v1_model.invoke("Say hello in Spanish, one word.")
    print("content type:", type(v1_answer.content).__name__)
    print("content     :", v1_answer.content)

**You should now see** `content type: list` and a list like `[{'type': 'reasoning', ...}, {'type': 'text', 'text': 'Hola'}]`: the standard blocks, stored. Without `output_version="v1"` the same call returns a `str`.

---
## Part 10 · 🔎 Research checkpoint: who supports `content_blocks` natively?

The module's question: *which providers currently support standard `content_blocks` natively, and is `langchain-groq` one of them?*

"Natively" can mean two things, so investigate both:

1. **Translation:** does `langchain-core` know how to translate this provider's messages into standard blocks? The translators live in `langchain_core.messages.block_translators`, one module per provider. Let's list them from the installed code.
2. **Input:** does the integration accept standard blocks as *input* (Part 8 showed `ChatGroq` converting a standard image block)?

In [ ]:
# 🧪 Offline: list the provider translators shipped in YOUR installed langchain-core.
import pkgutil

import langchain_core
from langchain_core.messages import block_translators

names = sorted(m.name for m in pkgutil.iter_modules(block_translators.__path__))
print(f"langchain-core {langchain_core.__version__} translators:", names)
print("Groq translator present:", "groq" in names)

**You should now see** (with `langchain-core` 1.6.9): `anthropic, bedrock, bedrock_converse, google_genai, google_vertexai, groq, langchain_v0, openai`, and `Groq translator present: True`.

**Finish the research and log it** in [`docs/research-log.md`](../../docs/research-log.md) (template at the top of that file):

- Open the [translators folder on GitHub](https://github.com/langchain-ai/langchain/tree/master/libs/core/langchain_core/messages/block_translators). Is the list on `master` the same as in your installed version?
- Read `groq.py` there. Which Groq fields become which blocks? (Hint: `reasoning_content` → `reasoning`; `executed_tools` → `server_tool_call` / `server_tool_result`, used by Groq's built-in browser search in Module 05.)
- Check the [chat integrations table](https://docs.langchain.com/oss/python/integrations/chat/index): is there a column for standard content? Note what you find, including if the docs **don't** say.
- Write your conclusion in one sentence, with the date and the package versions.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `KeyError` reading `message.content[0]["text"]` | `content` is a `str` for this provider | Use `.text` or `content_blocks` |
| No reasoning in `content_blocks` | Reasoning hidden: `reasoning_format="hidden"`, `include_reasoning=False`, or Qwen's `reasoning_effort="default"/"none"` | Check the model's reasoning options on [Groq reasoning](https://console.groq.com/docs/reasoning) |
| `<think>...</think>` inside `.text` | Qwen with `reasoning_format="raw"` (Groq's default for non-gpt-oss models) | Set `reasoning_format="parsed"` |
| 400 error mentioning `tool_call_id` | `ToolMessage` doesn't match any tool call id in the previous `AIMessage` | Copy the id from `ai_message.tool_calls[i]["id"]` |
| 400 error with an image | More than 3 images, unsupported format, or a non-vision model | One vision model, ≤3 images, PNG/JPEG |
| Input tokens explode in long chats | Whole history resent every turn | Trim or summarize (Module 03) |

## ⚠️ Production notes

> ⚠️ **Production note: storing conversations.**
> Part 7 writes the whole conversation to JSON. Real chats contain personal data. In production: store only what you need, encrypt at rest, set a retention period, redact PII before logging (Module 04), and record which model and prompt version produced each `AIMessage` (`response_metadata`, LangSmith metadata).

## 🏋️ Exercises

1. **(Easy)** Rebuild the Part 1 conversation using only dicts, then using only tuples. Confirm with `convert_to_messages` that you get the same objects.
2. **(Easy)** Add a fourth fake provider message to Part 4: Gemini returns thinking as `{"type": "thinking", "thinking": "..."}` blocks with `response_metadata={"model_provider": "google_genai"}`. Does `describe_blocks` still work unchanged?
3. **(Medium)** Write `last_reasoning(messages) -> str | None` that returns the reasoning text of the most recent `AIMessage` in a list, using only `content_blocks`. Test it on `[HumanMessage("hi"), groq_style]` and on a list without reasoning.
4. **(Stretch)** With a key: send the chart **twice** in one message (two image blocks) and ask whether the two images are identical. Compare `input_tokens` with Part 8. Then try 4 images and read the error with `explain_model_error`.

<details>
<summary><b>Show solutions</b></summary>

**2.**
```python
gemini_style = AIMessage(content=[{"type": "thinking", "thinking": "~4 chars per token."},
                                  {"type": "text", "text": "About 1.3 tokens per word."}],
                         response_metadata={"model_provider": "google_genai"})
print(*describe_blocks(gemini_style), sep="\n")
```
Expect a `reasoning` line and a `text` line: the Gemini translator maps `thinking` to `reasoning`. That's the point of standard blocks: new providers, same loop.

**3.**
```python
def last_reasoning(messages) -> str | None:
    """Return the reasoning text of the most recent AIMessage, or None."""
    for message in reversed(messages):
        if isinstance(message, AIMessage):
            parts = [b.get("reasoning", "") for b in message.content_blocks if b["type"] == "reasoning"]
            return "\n".join(parts) or None
    return None

print(last_reasoning([HumanMessage("hi"), groq_style]))
print(last_reasoning([HumanMessage("hi"), AIMessage("no thinking here")]))   # None
```

**4.** Two images should cost about 2 × 2,048 extra input tokens. Four images exceed Groq's limit of 3: expect a 400 error, classified as `invalid_request`.
</details>

## ✅ Checkpoint

- [ ] I can name the four message types and when each one is written, and by whom.
- [ ] I can explain `content` vs `content_blocks` and wrote one loop that reads reasoning from three providers.
- [ ] I built a tool-call round trip and know why `tool_call_id` must match.
- [ ] I serialized a conversation losslessly and in OpenAI format.
- [ ] (With a key) the vision model described the chart; input tokens included 2,048 for the image.
- [ ] I logged the content-blocks research finding in `docs/research-log.md`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## Summary

- Messages are the unit of context: role + content + metadata. Objects in code, dicts at the edges.
- `content_blocks` gives a standard, typed view of any provider's output; `.text` for just the answer.
- Tool calls live on `AIMessage.tool_calls`; results go back as `ToolMessage` with the matching id. `artifact` stays out of the prompt.
- Chat APIs are stateless: you resend history, and input tokens grow every turn.
- Images are just another content block; know the provider's limits and token cost.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 01.03 · Prompting fundamentals for open-weight models** → [`01-03-prompting-fundamentals.ipynb`](01-03-prompting-fundamentals.ipynb).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)